### 🚀 Run this notebook in [RenkuLab](https://renkulab.io/)

👉 [![launch - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/meteoswiss/opendata-nwp-demos/sessions/01KME52HC2FZ6ZHB30SSFG08PW/start)

1. Wait for the session to start
2. Navigate to `opendata-nwp-demos/09_constant_parameters.ipynb` and run the notebook without any further installation.

# Accessing constant parameters

This notebook demonstrates the retrieval of **constant parameters** such as grid information that are not part of the forecast GRIB files. The data is provided by MeteoSwiss as part of Switzerland’s [Open Government Data (OGD) initiative](https://www.meteoswiss.admin.ch/services-and-publications/service/open-data.html).



---

## 🔍 **What You’ll Do in This Notebook**

- 📥 Retrieve the vertical constant parameter HHL for ICON-CH1 and ICON-CH2
- 📥 Retrieve horizontal constant parameters (CLON, CLAT, FR_LAND, HSURF, SOILTYP) for ICON-CH2
- ✅ Verify grid consistency with the ICON-CH2 forecast using GRIB metadata (uuidOfHGrid)


> ⚠️ **Warning**: The constant parameters are only considered constant per run! Although they are not expected to change often, we recommend you check that the 'uuidOfHGrid' agrees with the forecast data, especially if you cache the constants.

---

## 📥 Retrieve constant parameters
The constant parameters are separated into horizontal and vertical constants, and available for both the ICON-CH1 and ICON-CH2 collections.

First, the urls are obtained:

In [ ]:
from meteodatalab import ogd_api

# Vertical
url_ch1_vert = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch1",
    asset_id="vertical_constants_icon-ch1-eps.grib2"
)
url_ch2_vert = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch2",
    asset_id="vertical_constants_icon-ch2-eps.grib2"
)

# Horizonal
url_ch1_hor = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch1",
    asset_id="horizontal_constants_icon-ch1-eps.grib2"
)
url_ch2_hor = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch2",
    asset_id="horizontal_constants_icon-ch2-eps.grib2"
)

Next, the data can be fetched. HHL is the only vertical constant, referring to the height of the model half levels. For more information, see the [model grid documentation](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#vertical-grid).

> 💡 **Tip**: The HHL parameter is also used in the notebook [05_interpolate_vertically.ipynb](05_interpolate_vertically.ipynb).

In [ ]:
from meteodatalab import grib_decoder, data_source

# load CH2 vertical constants
ds_ch2_vert = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url_ch2_vert]),
    request={"param": "HHL"},
    geo_coords=lambda uuid: {}
)

# load CH1 vertical constants
ds_ch1_vert = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url_ch1_vert]),
    request={"param": "HHL"},
    geo_coords=lambda uuid: {}
)

ds_ch1_vert["HHL"]

In the horizontal constants, CLON, CLAT, FR_LAND, HSURF and SOILTYP are available:
| Name      | Description                                                                      |
|-----------|----------------------------------------------------------------------------------|
| CLON      | Longitude of the center point coordinate of each triangle on the horizontal grid |
| CLAT      | Latitude of the center point coordinate of each triangle on the horizontal grid  |
| FR_LAND   | Land cover indicator (1 for land, 0 for sea)                                     |
| HSURF     | Geometric height of the earth's surface above sea level                          |
| SOILTYPE  | Type of soil, based on the [soilType.table](https://github.com/COSMO-ORG/eccodes-cosmo-resources/blob/master/definitions/soilType.table)                                                                |

In [ ]:
# load CH2 horizontal constants
ds_ch2_hor = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url_ch2_hor]),
    request={"param": ["CLON", "CLAT", "FR_LAND", "HSURF", "SOILTYP"]},
    geo_coords=lambda uuid: {}
)

In [ ]:
ds_ch2_hor.keys()

## ✅ Verify grid consistency

The grib metadata carries a lot of information:

In [ ]:
# show all metadata
ds_ch2_vert["HHL"].metadata.dump()

A universally unique identifier (UUID) of the horizontal grid the data is associated with is provided. You can see that the ICON-CH1 and ICON-CH2 models use a different grid:

In [ ]:
print("CH1 HHL    : ", ds_ch1_vert["HHL"].metadata.get("uuidOfHGrid"))
print("CH2 HHL    : ", ds_ch2_vert["HHL"].metadata.get("uuidOfHGrid"))
print("CH2 SOILTYP: ", ds_ch2_hor["SOILTYP"].metadata.get("uuidOfHGrid"))

Finally, we fetch the 2m temperature from the ICON-CH2 control forecast and compare the UUID of the grid with the UUID obtained from the constants.

> ⚠️ **Warning**: The constant parameters are only considered constant per run! Although they are not expected to change often, we recommend you check that the 'uuidOfHGrid' agrees with the forecast data, especially if you cache the constants.

In [ ]:
from meteodatalab import ogd_api
from earthkit.data import config
config.set("cache-policy", "temporary")

da_T2M_ch2 = ogd_api.get_from_ogd(
    ogd_api.Request(
        collection="ogd-forecasting-icon-ch2",
        variable="T_2M",
        ref_time="latest",
        perturbed=False,
        lead_time=f"P0DT0H",
    )
)

In [ ]:
assert da_T2M_ch2.metadata.get("uuidOfHGrid") == ds_ch2_vert["HHL"].metadata.get("uuidOfHGrid")